# 智能指针
智能指针都实现了Deref和Drop特征

# Box堆对象分配
## Rust中的堆栈
栈内存从高地址向下增长，且栈内存时连续分配的。操作系统堆栈内存大小是有限制的。Rust中，main线程的栈大小是8MB， 普通线程是2MB。
堆内存是从低地址向上增长的，堆内存通常只受物理内存的限制，而且通常是不连续的。
### 堆栈的性能
1. 小型数据，在栈上的分配性能和读取性能都要比堆上高
2. 中型数据，栈上分配性能高，但读取性能和堆上无区别，**因为无法利用寄存器或CPU高速缓存，最终还是要经过一次内存寻址**
3. 大型数据，只建议在堆上分配和使用

## Box的使用场景
Box只是简单的封装，将值存储在堆上。一般使用场景如下：
1. 特意将数据分配在内存上
2. 数据较大时，有不想再转移所有权时进行数据拷贝
3. 类型的大小在编译时无法确定，但我们又需要固定大小的类型时
4. 特征对象，用于说明实现一个特征，而不是某个特定的类型

#### 使用Box将数据存在堆上
```rust
let a = Box::new(3);
println!("a={}",a);

//let b = a+1; 报错
```
* 在打印时，隐式调用了Deref特征对智能指针进行解引用
* 在表达式中，无法自动隐式的执行deref解引用操作，需要显示的调用*操作来进行解引用
* 在作用域结束时，Box会自动调用Drop特征将相关资源释放。

#### 避免栈上数据拷贝
<font color=red>栈上数据转移所有权时，实际上是把数据拷贝了一份</font>，最终新旧变量各自拥有不同的数据，因此所有权并未转移。(所以自定义结构体栈上分配一般要实现Copy特征)
而堆上，底层数据不会拷贝，转移所有权仅仅是复制一份栈上的指针，再将新的指针赋予新的变量，然后让拥有旧指针的变量失效

#### 将动态大小类型变为Sized固定大小类型
Rust需要再编译时知道类型占用空间大小，如果一种类型在编译时无法知道具体的大小，那么被称为动态大小类型DST。 树和图就是这种类型。
```rust
enum List {
    Cons(i32, List),
    Nil,
}

enum List {
    Cons(i32, Box<List>),
    Nil,
}
```
#### 特征对象
想实现不同类型组成的数组有两个方法：枚举和特征对象
<font color=red>特征对象需要使用dyn进行修饰</font>

## Box内存布局


## Box::leak
Box::leak 是Box的一个关联函数，它可以消费掉Box并强制目标值从内存中泄露。
可以把String类型，编程一个'static生命周期的&str类型
```rust
fn main() {
    let s = gen_static_str();
    println!("{}", s);
}

fn gen_static_str()->&'static str {
    let mut s = String::new();
    s.push_str("hello world");

    Box::leak(s.into_boxed_str())
}
```

# Deref 解引用


# Drop释放资源

# RC与Arc
* 在图数据结构中，多个变可能会拥有同一个节点，该节点直到没有边指向它时， 才应该被释放
* 在多线程中，多个线程可能会持有同一数据，但受限于Rust的安全机制， 无法同时获取该数据的可变引用

Rust在所有权机制之外引入了 通过计数的方式，允许一个数据资源在同一时刻拥有多个所有者

<font color=red> RC适用于单线程， Arc适用于多线程</font>

## Rc
当我们希望在堆上分配一个对象供程序的多个部分使用且无法确定哪个部分最后一个结束时，就可以使用Rc成为数据值的所有者。
```Rust
use std::rc::Rc
fn main() {
    let a = Rc::new(String::from("hello, world"));
    let b = Rc::clone(&a);
    assert_eq!(2, Rc::Strong_count(&a));
    assert_eq!(Rc::strong_count(&a), Rc::strong_count(&b));
}
```
***Rc::clone***:
Rc::clone 只是克隆了一份智能指针Rc。 这里的clone仅仅复制了智能指针并增加计数引用，没有克隆底层数据

**Rc指向底层数据的不可变引用，无法通过它来修改数据**
<font color=ref>Rc没有实现Send特征， 且Rc需要管理引用计数，但该引用计数没有使用任何并发原语，因此无法实现原子化的计数操作， 因此无法用于多线程</font>

#### Rc简单总结
* Rc/Arc是不可变引用，智能读取，无法修改它所指向的值
* 一旦最后一个拥有者消失，资源将会被回收， 这个生命周期是在编译的时候确定下来的
* Rc智能用于同一线程内部，想要线程之间共享对象，要使用Arc
* Rc是一个智能指针，实现了Deref特征

## Arc
Arc 是Atomic Rc的缩写
Arc和Rc拥有完全一样的API, 只是引用的包不一样
```rust
use std::sync::Arc;
use std::thread;
fn main() {
    let s = Arc::new(String::from("多线程"));
    for _ in 0..10 {
        let s = Arc::clone(&s);
        let handle = thread::spawn(move || {
            println!("{}", s)
        })
    }
}
```


# Cell 和 RefCell
Rust提供了Cell和RefCell 用于内部可变性。 即**在拥有不可变引用的同时修改目标数据**
> 内部可变性的实现是因为Rust使用了unsafe来做到这一点， 但对于使用者来说，这些都是透明的，因为这些不安全代码都被封装到安全的API中

## Cell
Cell和RefCell在功能上没有区别，区别在于<font color=red>Cell适用于T实现Copy的情况, 因为get方法需要</font>
```rust
use std::cell::Cell
fn main() {
    let c = Cell::new("asdf);
    let one = c.get();
    c.set("qwer");
    let two = c.get();
    println!("{}, {}", one, two);
    let d = Cell::new(String::from("asdf"));
}
```
使用get 获取数据， 使用后set 设置数据
